# Tipos de Capas en Redes Neuronales


En este notebook veremos los **bloques de construcción** de una red neuronal densa
(también llamada *feedforward* o *fully connected*).

Para cada tipo de capa responderemos a tres preguntas:
1. **¿Qué hace?**
2. **¿Cuándo se usa?**
3. **¿Cómo se escribe en Keras?**

Una red neuronal es básicamente una **pila de capas**. Los datos entran por arriba,
atraviesan cada capa una tras otra, y salen como predicción por abajo. Elegir bien
qué capas apilar (y en qué orden) es gran parte del trabajo.

## Preparación

Importamos Keras y creamos unos datos de sinteticos para los ejemplos.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

print('TensorFlow', tf.__version__)
np.set_printoptions(suppress=True, precision=12)


X = np.random.rand(500, 8).astype('float32')
y = np.random.randint(0, 3, 500)
print('X:', X.shape, ' y:', y.shape)

TensorFlow 2.19.0
X: (500, 8)  y: (500,)


In [6]:
X

array([[0.6100065  , 0.58307904 , 0.23498617 , ..., 0.3439932  ,
        0.50130993 , 0.76845276 ],
       [0.55210686 , 0.23586115 , 0.95057064 , ..., 0.085131675,
        0.41724038 , 0.23019458 ],
       [0.1290547  , 0.120542265, 0.55100703 , ..., 0.16876665 ,
        0.004877017, 0.007191424],
       ...,
       [0.66301334 , 0.8032194  , 0.043751795, ..., 0.39364502 ,
        0.16953076 , 0.80789727 ],
       [0.531798   , 0.8651936  , 0.46803564 , ..., 0.8261222  ,
        0.25418323 , 0.20369804 ],
       [0.5485597  , 0.06619704 , 0.35661063 , ..., 0.48644713 ,
        0.9474337  , 0.15022428 ]], dtype=float32)

---
## 1. Capa de entrada — `Input`

**¿Qué hace?**
No realiza ningún cálculo. Solo le dice a la red **qué forma tienen tus datos**
(cuántas características entran). Es la puerta de entrada.

**¿Cuándo se usa?**
Siempre, al principio de la red. Es obligatoria (aunque a veces se declara de forma
implícita dentro de la primera capa con el argumento `input_shape`).

**Regla práctica:** el tamaño de la entrada lo marcan tus datos, no tú.
Si cada muestra tiene 8 valores → `Input(shape=(8,))`.

In [ ]:
keras.Input(shape=(8,))

In [ ]:
layers.Dense(16, activation='relu', input_shape=(8,))

---
## 2. Capa densa — `Dense` (la más importante)

**¿Qué hace?**
Conecta **todas** las neuronas de la capa anterior con **todas** las de esta capa
(por eso se llama *fully connected*). Es donde la red realmente "aprende":
cada conexión tiene un peso que se ajusta durante el entrenamiento.

**¿Cuándo se usa?**
Es la capa principal de toda red densa. La usarás en casi todas las capas:
ocultas y de salida.

**El número de neuronas (unidades):**
- Pocas neuronas → la red aprende poco (riesgo de *underfitting*).
- Muchas neuronas → la red aprende mucho, pero más lento y con riesgo de *overfitting*.
- Valores habituales en capas ocultas: 16, 32, 64, 128, 256.

**Regla práctica:** empieza con 1-2 capas densas de 32-64 neuronas y ajusta según resultados.

In [7]:
modelo = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(64, activation='relu'), 
    layers.Dense(32, activation='relu'),   
    layers.Dense(3,  activation='softmax') 
])
modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,755 (10.76 KB)

 Trainable params: 2,755 (10.76 KB)

 Non-trainable params: 0 (0.00 B)

---
## 3. Funciones de activación

**¿Qué hacen?**
Deciden si una neurona "se enciende" y cuánto. Sin ellas, apilar capas no serviría
de nada: la red solo podría aprender relaciones en línea recta. La activación es lo
que permite aprender patrones complejos.

### Cuál elegir según dónde esté la capa

| Función | Dónde se usa | Qué hace |
|---------|--------------|----------|
| **ReLU** | Capas ocultas (por defecto) | Deja pasar positivos, bloquea negativos |
| **Sigmoid** | Salida binaria (sí/no) | Comprime a un valor entre 0 y 1 |
| **Softmax** | Salida multiclase | Reparte el 100% entre las clases |
| **Tanh** | Alternativa en ocultas | Comprime entre -1 y 1 |
| **Lineal** (sin activación) | Salida de regresión | Deja el número tal cual |

**Regla práctica:** ReLU en las capas ocultas, y la activación de la salida la decide
el tipo de problema (lo vemos en la sección 7).

In [8]:
layers.Dense(32, activation='relu')

<Dense name=dense_3, built=False>

---
## 4. Capa de aplanado — `Flatten`

**¿Qué hace?**
Convierte datos con varias dimensiones en un **vector plano** (1D). Por ejemplo,
una imagen de 28×28 se convierte en una fila de 784 valores.

**¿Cuándo se usa?**
Cuando tus datos de entrada tienen más de una dimensión pero quieres usar capas
`Dense` (que solo entienden vectores planos). Se pone justo después de la entrada.

**Regla práctica:** si tus datos ya son una tabla de filas y columnas (datos
tabulares), normalmente NO lo necesitas. Si son imágenes y quieres una red densa,
sí lo necesitas al principio.

In [10]:
modelo_flat = keras.Sequential([
    keras.Input(shape=(28, 28)),   
    layers.Flatten(),             
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
modelo_flat.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

---
## 5. Capa de regularización — `Dropout`

**¿Qué hace?**
Durante el entrenamiento, **apaga al azar** un porcentaje de neuronas en cada paso.
Esto obliga a la red a no depender de neuronas concretas y la hace más robusta.

**¿Cuándo se usa?**
Cuando detectas **overfitting** (la red va muy bien en entrenamiento pero mal en
validación). Se coloca *entre* capas densas.

**El valor (rate):** es la fracción que se apaga.
- `0.2` apaga el 20% · `0.5` apaga el 50%.
- Rango típico: entre 0.2 y 0.5.

**Importante:** el dropout solo actúa al entrenar. Al predecir, todas las neuronas
están activas (Keras lo gestiona automáticamente).

**Regla práctica:** no lo pongas "por si acaso". Añádelo si ves overfitting.

In [11]:
modelo_dropout = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),               
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
])
modelo_dropout.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,755 (10.76 KB)

 Trainable params: 2,755 (10.76 KB)

 Non-trainable params: 0 (0.00 B)

---
## 6. Capa de normalización — `BatchNormalization`

**¿Qué hace?**
Mantiene los valores que circulan entre capas en un rango "saludable"
(ni demasiado grandes ni demasiado pequeños). Esto ayuda a que la red entrene
más rápido y de forma más estable.

**¿Cuándo se usa?**
En redes que entrenan lento o de forma inestable. Se coloca entre la capa densa
y su activación (o justo después). Es opcional en redes pequeñas.

**Regla práctica:** en redes densas sencillas muchas veces no hace falta.
Es más útil cuando la red empieza a crecer. Para empezar, no te obsesiones con ella.

In [13]:
modelo_bn = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')
])
modelo_bn.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,011 (11.76 KB)

 Trainable params: 2,883 (11.26 KB)

 Non-trainable params: 128 (512.00 B)

---
## 7. La capa de salida — cómo elegirla

La última capa es siempre una `Dense`, pero su **número de neuronas** y su
**activación** dependen del tipo de problema. Esta es la decisión más importante
y la que más errores causa en principiantes.

| Tipo de problema | Neuronas de salida | Activación | Loss recomendada |
|------------------|--------------------|------------|------------------|
| Binario (sí/no) | 1 | `sigmoid` | `binary_crossentropy` |
| Multiclase (1 etiqueta) | nº de clases | `softmax` | `sparse_categorical_crossentropy` |
| Regresión (un número) | 1 | ninguna (lineal) | `mse` |

**Regla práctica:** primero pregunta "¿qué quiero predecir?" y de ahí sale la salida.

In [12]:
salida_binaria = layers.Dense(1, activation='sigmoid')
salida_multiclase = layers.Dense(10, activation='softmax')
salida_regresion = layers.Dense(1)  

---
## 8. Ejemplo completo: juntándolo todo

Montamos una red densa con varias de las capas vistas y la entrenamos con
nuestros datos de juguete (3 clases).

In [14]:
modelo_final = keras.Sequential([
    keras.Input(shape=(8,)),              # 1. entrada: 8 caracteristicas
    layers.Dense(64, activation='relu'),  # 2. capa oculta + ReLU
    layers.Dropout(0.3),                  # 3. regularizacion
    layers.Dense(32, activation='relu'),  # 4. capa oculta + ReLU
    layers.Dropout(0.3),                  # 5. regularizacion
    layers.Dense(3, activation='softmax') # 6. salida: 3 clases
])

modelo_final.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo_final.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_17 (Dense)                │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,755 (10.76 KB)

 Trainable params: 2,755 (10.76 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
historia = modelo_final.fit(X, y, epochs=5, batch_size=32,
                            validation_split=0.2, verbose=1)

Epoch 1/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3904 - loss: 1.0811 - val_accuracy: 0.3000 - val_loss: 1.1196
Epoch 2/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3698 - loss: 1.0963 - val_accuracy: 0.3000 - val_loss: 1.1190
Epoch 3/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3721 - loss: 1.0940 - val_accuracy: 0.3000 - val_loss: 1.1195
Epoch 4/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4130 - loss: 1.0791 - val_accuracy: 0.3000 - val_loss: 1.1191
Epoch 5/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3467 - loss: 1.1012 - val_accuracy: 0.2900 - val_loss: 1.1195


---
## 9. Tabla resumen: ¿cuándo usar cada capa?

| Capa | Para qué sirve | ¿Cuándo la pongo? |
|------|----------------|-------------------|
| **Input** | Define la forma de los datos | Siempre, al principio |
| **Dense** | El cálculo principal, donde se aprende | En casi todas las capas |
| **Activación** (ReLU...) | Permite aprender patrones complejos | Siempre tras una Dense oculta |
| **Flatten** | Aplana datos multidimensionales | Si la entrada no es un vector plano |
| **Dropout** | Evita el overfitting | Solo si hay overfitting |
| **BatchNormalization** | Estabiliza y acelera el entrenamiento | Si la red entrena mal/lento |
| **Dense de salida** | Produce la predicción final | Siempre, al final |

### Plantilla mental para una red densa

```
Input  →  [ Dense + ReLU  (+ Dropout) ] × N  →  Dense de salida
```

Donde **N** es el número de capas ocultas (empieza con 1 o 2) y el Dropout es
opcional (solo si lo necesitas).

### Los 3 errores más comunes
1. Olvidar que la **salida** depende del problema (binario / multiclase / regresión).
2. Poner **Dropout muy alto** o en todas partes "por si acaso".
3. Empezar con una red **enorme** para un problema simple.